In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.fct_daily_employee_utilization AS
WITH 
-- 1. Wykrycie dni, w których cokolwiek zaplanowano lub zarejestrowano
distinct_active_dates AS (
  SELECT start_date AS activity_date FROM data_warehouse_factory.gold.fct_final_operator_assignments
  UNION
  SELECT production_date AS activity_date FROM data_warehouse_factory.gold.fct_events_daily
),

-- 2. Pobranie dat kalendarzowych tylko dla aktywnych dni
calendar_days AS (
  SELECT 
    d.date_key,
    d.date AS production_date
  FROM data_warehouse_factory.gold.dim_date d
  INNER JOIN distinct_active_dates a
    ON d.date = a.activity_date
),

-- 3. Siatka pracowników zatrudnionych w tych konkretnych dniach (norma 8.0h)
employee_days_capacity AS (
  SELECT 
    d.date_key,
    d.production_date,
    e.employee_key,
    e.employee_full_name,
    8.0 AS total_capacity_hours
  FROM calendar_days d
  CROSS JOIN data_warehouse_factory.gold.dim_employees e
  WHERE e.date_of_employment <= d.production_date
    AND (e.date_of_leaving IS NULL OR e.date_of_leaving >= d.production_date)
),

-- 4. SUMA godzin produkcyjnych (Plan domyślny + Wszystkie zrealizowane Swapy)
planned_hours_worked AS (
  SELECT 
    date_key,
    assigned_operator_key AS employee_key,
    ROUND(SUM(duration_minutes) / 60.0, 2) AS planned_production_hours
  FROM data_warehouse_factory.gold.fct_final_operator_assignments
  WHERE assigned_operator_key IS NOT NULL
  GROUP BY date_key, assigned_operator_key
),

-- 5. Rozbicie dodatkowych aktywności oraz absencji
event_hours AS (
  SELECT 
    e.date_key,
    e.employee_key,
    -- Szkolenia, przezbrojenia maszyny itp. (bez swapów, bo są już w pkt 4)
    ROUND(SUM(CASE WHEN d.is_utilized_time = TRUE AND coalesce(d.is_swap, FALSE) = FALSE THEN e.duration_minutes ELSE 0 END) / 60.0, 2) AS additional_activity_hours,
    -- Absencje (L4, urlopy)
    ROUND(SUM(CASE WHEN d.is_absence = TRUE THEN e.duration_minutes ELSE 0 END) / 60.0, 2) AS absence_hours
  FROM data_warehouse_factory.gold.fct_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  GROUP BY e.date_key, e.employee_key
),

-- 6. Wyliczenie bilansu dziennego
calculated_utilization AS (
  SELECT 
    c.date_key,
    c.production_date,
    c.employee_key,
    c.employee_full_name,
    c.total_capacity_hours,
    
    COALESCE(p.planned_production_hours, 0.0) AS planned_production_hours,
    COALESCE(eh.additional_activity_hours, 0.0) AS additional_activity_hours,
    COALESCE(eh.absence_hours, 0.0) AS absence_hours,
    
    -- Czas efektywnie przepracowany (Produkcja + Zadania dodatkowe, max 8.0h)
    LEAST(
      c.total_capacity_hours, 
      COALESCE(p.planned_production_hours, 0.0) + COALESCE(eh.additional_activity_hours, 0.0)
    ) AS total_utilized_hours,
    
    -- Rzeczywisty wolny czas (Norma 8h minus praca, zadania i absencje)
    GREATEST(
      0.0, 
      c.total_capacity_hours - (
        COALESCE(p.planned_production_hours, 0.0) + 
        COALESCE(eh.additional_activity_hours, 0.0) + 
        COALESCE(eh.absence_hours, 0.0)
      )
    ) AS unutilized_hours
  FROM employee_days_capacity c
  LEFT JOIN planned_hours_worked p 
    ON c.date_key = p.date_key AND c.employee_key = p.employee_key
  LEFT JOIN event_hours eh 
    ON c.date_key = eh.date_key AND c.employee_key = eh.employee_key
)

SELECT 
  md5(concat_ws('||', cast(date_key as string), cast(employee_key as string))) AS utilization_key,
  date_key,
  production_date,
  employee_key,
  employee_full_name,
  total_capacity_hours,
  planned_production_hours,
  additional_activity_hours,
  absence_hours,
  total_utilized_hours,
  unutilized_hours,
  ROUND((total_utilized_hours / total_capacity_hours) * 100, 2) AS utilization_pct,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM calculated_utilization;